<a href="https://colab.research.google.com/github/dhanushkumar-amk/BUILD-OWN-XYZ/blob/main/BPE_Tokenizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Prepare corpus: split into words, count frequencies, split into characters**

In [9]:
from collections import defaultdict, Counter

def get_corpus_stats(corpus):
    """
    Takes a list of sentences, returns a dict:
    word (as tuple of chars + </w>) -> frequency count
    """
    word_freq = Counter()
    for sentence in corpus:
        words = sentence.lower().split()
        for word in words:
            word_freq[word] += 1

    # Convert each word into a tuple of characters + end-of-word marker
    vocab = {}
    for word, freq in word_freq.items():
        chars = tuple(word) + ("</w>",)
        vocab[chars] = freq

    return vocab

# Test
corpus = [
    "low low low low low",
    "lower lower",
    "newest newest newest newest newest newest",
    "widest widest widest"
]
vocab = get_corpus_stats(corpus)
for word, freq in vocab.items():
    print(word, "->", freq)

('l', 'o', 'w', '</w>') -> 5
('l', 'o', 'w', 'e', 'r', '</w>') -> 2
('n', 'e', 'w', 'e', 's', 't', '</w>') -> 6
('w', 'i', 'd', 'e', 's', 't', '</w>') -> 3


## **Count pair frequencies**

In [10]:
def get_pair_freqs(vocab):
    """
    Counts frequency of every adjacent symbol pair across the vocab
    """
    pairs = defaultdict(int)
    for word, freq in vocab.items():
        for i in range(len(word) - 1):
            pair = (word[i], word[i + 1])
            pairs[pair] += freq
    return pairs

# Test
pairs = get_pair_freqs(vocab)
for pair, freq in sorted(pairs.items(), key=lambda x: -x[1]):
    print(pair, "->", freq)

('e', 's') -> 9
('s', 't') -> 9
('t', '</w>') -> 9
('w', 'e') -> 8
('l', 'o') -> 7
('o', 'w') -> 7
('n', 'e') -> 6
('e', 'w') -> 6
('w', '</w>') -> 5
('w', 'i') -> 3
('i', 'd') -> 3
('d', 'e') -> 3
('e', 'r') -> 2
('r', '</w>') -> 2


## **Merge the most frequent pair**

In [11]:
def merge_vocab(pair, vocab):
    """
    Merges all occurrences of `pair` in the vocab into one symbol
    """
    new_vocab = {}
    bigram = pair
    replacement = "".join(pair)

    for word, freq in vocab.items():
        new_word = []
        i = 0
        while i < len(word):
            # Check if current position matches the pair
            if i < len(word) - 1 and word[i] == bigram[0] and word[i+1] == bigram[1]:
                new_word.append(replacement)
                i += 2
            else:
                new_word.append(word[i])
                i += 1
        new_vocab[tuple(new_word)] = freq

    return new_vocab

# Test: find and merge the top pair once
best_pair = max(pairs, key=pairs.get)
print("Merging:", best_pair)

vocab = merge_vocab(best_pair, vocab)
for word, freq in vocab.items():
    print(word, "->", freq)

Merging: ('e', 's')
('l', 'o', 'w', '</w>') -> 5
('l', 'o', 'w', 'e', 'r', '</w>') -> 2
('n', 'e', 'w', 'es', 't', '</w>') -> 6
('w', 'i', 'd', 'es', 't', '</w>') -> 3


## **Full iterative training loop**

In [12]:
def train_bpe(corpus, num_merges=15):
    vocab = get_corpus_stats(corpus)
    merge_rules = []  # ordered list of merges (Step 7)

    for i in range(num_merges):
        pairs = get_pair_freqs(vocab)
        if not pairs:
            break  # nothing left to merge

        best_pair = max(pairs, key=pairs.get)
        vocab = merge_vocab(best_pair, vocab)
        merge_rules.append(best_pair)

        print(f"Merge {i+1}: {best_pair} -> {''.join(best_pair)}")

    return vocab, merge_rules

# Test — train on the corpus
corpus = [
    "low low low low low",
    "lower lower",
    "newest newest newest newest newest newest",
    "widest widest widest"
]

final_vocab, merge_rules = train_bpe(corpus, num_merges=15)

print("\nFinal word representations:")
for word, freq in final_vocab.items():
    print(word, "->", freq)

print("\nMerge rules (in order):")
print(merge_rules)

Merge 1: ('e', 's') -> es
Merge 2: ('es', 't') -> est
Merge 3: ('est', '</w>') -> est</w>
Merge 4: ('l', 'o') -> lo
Merge 5: ('lo', 'w') -> low
Merge 6: ('n', 'e') -> ne
Merge 7: ('ne', 'w') -> new
Merge 8: ('new', 'est</w>') -> newest</w>
Merge 9: ('low', '</w>') -> low</w>
Merge 10: ('w', 'i') -> wi
Merge 11: ('wi', 'd') -> wid
Merge 12: ('wid', 'est</w>') -> widest</w>
Merge 13: ('low', 'e') -> lowe
Merge 14: ('lowe', 'r') -> lower
Merge 15: ('lower', '</w>') -> lower</w>

Final word representations:
('low</w>',) -> 5
('lower</w>',) -> 2
('newest</w>',) -> 6
('widest</w>',) -> 3

Merge rules (in order):
[('e', 's'), ('es', 't'), ('est', '</w>'), ('l', 'o'), ('lo', 'w'), ('n', 'e'), ('ne', 'w'), ('new', 'est</w>'), ('low', '</w>'), ('w', 'i'), ('wi', 'd'), ('wid', 'est</w>'), ('low', 'e'), ('lowe', 'r'), ('lower', '</w>')]


## **Build the token vocabulary (base chars + merges) and ID mappings**

In [13]:
def build_bpe_vocabulary(corpus, merge_rules):
    # Collect base characters from training data
    base_chars = set()
    for sentence in corpus:
        for word in sentence.lower().split():
            base_chars.update(list(word))
    base_chars.add("</w>")

    # Collect merged symbols
    merged_symbols = set("".join(pair) for pair in merge_rules)

    # Combine everything
    special_tokens = ["<PAD>", "<START>", "<END>", "<UNK>"]
    all_tokens = special_tokens + sorted(base_chars) + sorted(merged_symbols)

    token_to_id = {token: idx for idx, token in enumerate(all_tokens)}
    id_to_token = {idx: token for token, idx in token_to_id.items()}

    return token_to_id, id_to_token, base_chars

token_to_id, id_to_token, base_chars = build_bpe_vocabulary(corpus, merge_rules)
print("Vocab size:", len(token_to_id))
print(token_to_id)

Vocab size: 30
{'<PAD>': 0, '<START>': 1, '<END>': 2, '<UNK>': 3, '</w>': 4, 'd': 5, 'e': 6, 'i': 7, 'l': 8, 'n': 9, 'o': 10, 'r': 11, 's': 12, 't': 13, 'w': 14, 'es': 15, 'est': 16, 'est</w>': 17, 'lo': 18, 'low': 19, 'low</w>': 20, 'lowe': 21, 'lower': 22, 'lower</w>': 23, 'ne': 24, 'new': 25, 'newest</w>': 26, 'wi': 27, 'wid': 28, 'widest</w>': 29}


## **Apply learned merges to new/unseen text**

In [14]:
def apply_bpe(word, merge_rules, base_chars):
    """
    Tokenizes a single word using the learned merge rules, in order.
    Falls back to <UNK> for characters never seen in training (Step 10).
    """
    # Step: split into characters, checking for unknown chars
    symbols = []
    for ch in word:
        if ch in base_chars:
            symbols.append(ch)
        else:
            symbols.append("<UNK>")  # Step 10 fallback
    symbols.append("</w>")

    # Apply each merge rule in order
    for pair in merge_rules:
        i = 0
        new_symbols = []
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == pair[0] and symbols[i+1] == pair[1]:
                new_symbols.append("".join(pair))
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1
        symbols = new_symbols

    return symbols

def bpe_tokenize(text, merge_rules, base_chars):
    words = text.lower().split()
    tokens = []
    for word in words:
        tokens.extend(apply_bpe(word, merge_rules, base_chars))
    return tokens

# Test on a word that WAS in training
print(bpe_tokenize("lowest", merge_rules, base_chars))

# Test on completely new text
print(bpe_tokenize("I love the lowest newest widest", merge_rules, base_chars))

['low', 'est</w>']
['i', '</w>', 'lo', '<UNK>', 'e', '</w>', 't', '<UNK>', 'e', '</w>', 'low', 'est</w>', 'newest</w>', 'widest</w>']


## **Encode / decode using the BPE vocabulary**

In [15]:
def bpe_encode(text, merge_rules, base_chars, token_to_id):
    tokens = bpe_tokenize(text, merge_rules, base_chars)
    ids = []
    for token in tokens:
        if token in token_to_id:
            ids.append(token_to_id[token])
        else:
            ids.append(token_to_id["<UNK>"])
    return ids

def bpe_decode(ids, id_to_token):
    tokens = [id_to_token[i] for i in ids]
    return " ".join(tokens)

# Test full round trip
text = "lowest newest"
encoded = bpe_encode(text, merge_rules, base_chars, token_to_id)
decoded = bpe_decode(encoded, id_to_token)

print("Original:", text)
print("Tokens:", bpe_tokenize(text, merge_rules, base_chars))
print("Encoded:", encoded)
print("Decoded:", decoded)

Original: lowest newest
Tokens: ['low', 'est</w>', 'newest</w>']
Encoded: [19, 17, 26]
Decoded: low est</w> newest</w>


## **Test with a genuinely novel character**

In [16]:
# A character never seen in training (like an emoji or foreign script)
test_text = "lowest 🍕"
tokens = bpe_tokenize(test_text, merge_rules, base_chars)
print("Tokens:", tokens)
# The emoji should map to <UNK> while "lowest" tokenizes normally

Tokens: ['low', 'est</w>', '<UNK>', '</w>']
